# Timeseries Analysis Notebook

**NOTE using the de-sar-sample-data envrionment for analysis**

In [ ]:
import fsspec
import xarray as xr
import rioxarray
from dotenv import load_dotenv
import re
import pandas as pd
import numpy as np
from scipy.ndimage import uniform_filter
import geopandas as gpd
from dea_tools.plotting import xr_animation
from dea_tools.temporal import xr_optical_flow, xr_regression # !pip install dea-tools==0.4.8dev13
from PIL import Image, ImageSequence
import matplotlib.pyplot as plt
import os
import json
from shapely.geometry import Polygon, mapping
import contextily as ctx
import cv2

## Set envrionment credentials for AWS access

In [ ]:
load_dotenv('/home/ec2-user/sar-pipeline/.env')

## Select the burst to assess

In [ ]:
# t007_014550_iw2 (scene 1) - slope and aspect
# t007_014551_iw2 (scene 1) - slope and aspect
# t007_014549_iw1 (scene 1) - elevation
# t007_014543_iw2 (scene 2) - elevation
burst = "t007_014549_iw1"
burst_shape = f"burst_shapefiles/{burst}.json"
burst_shape = gpd.read_file(burst_shape)

## Create a smaller ROI within burst for zoom

In [ ]:
burst_roi = False # if true, assess a zoomed area centered around below. Else assess whole burst.
if burst_roi:
    side_length = 10_000
    burst_roi_centers = {
        't007_014550_iw2' : (-1514900,-640900),
        't007_014551_iw2' : (-1515100,-660600),
        't007_014549_iw1' : (-1482400,-623700),
        't007_014543_iw2' : (-1514772,-508648),
    }

def save_square_geojson(center_x, center_y, side_length, filename):
    """
    Create a square polygon around a centre point (EPSG:3031) and save as GeoJSON.

    Parameters
    ----------
    center_x : float
        X coordinate (EPSG:3031)
    center_y : float
        Y coordinate (EPSG:3031)
    side_length : float
        Length of square's side (same units as EPSG:3031, e.g. metres)
    filename : str
        Output filename for the GeoJSON file

    Returns
    -------
    dict
        A GeoJSON FeatureCollection dictionary
    """
    
    half = side_length / 2.0

    # Define square corners (clockwise or counter-clockwise is fine)
    square = Polygon([
        (center_x - half, center_y - half),
        (center_x + half, center_y - half),
        (center_x + half, center_y + half),
        (center_x - half, center_y + half),
        (center_x - half, center_y - half),
    ])

    feature = {
        "type": "Feature",
        "geometry": mapping(square),
        "properties": {
            "center_x": center_x,
            "center_y": center_y,
            "side_length": side_length,
            "crs": "EPSG:3031"
        }
    }

    feature_collection = {
        "type": "FeatureCollection",
        "features": [feature]
    }

    # Save to file
    with open(filename, "w") as f:
        json.dump(feature_collection, f, indent=2)

    return feature_collection

if burst_roi:
    roi_x, roi_y = burst_roi_centers[burst]
    burst_roi_shape = f'burst_shapefiles/{burst}_{roi_x}_{roi_y}.json'
    save_square_geojson(roi_x, roi_y, side_length, burst_roi_shape)
    burst_roi_shape = gpd.read_file(burst_roi_shape)
    burst_roi_shape = burst_roi_shape.set_crs(epsg=3031, inplace=False,allow_override=True)
    
    # plot to see overlap
    fig, ax = plt.subplots(figsize=(8,8))
    burst_shape_3031 = burst_shape.to_crs(epsg=3031)
    burst_roi_shape.plot(ax=ax, color='red', alpha=0.5, edgecolor='black', label="ROI")
    burst_shape_3031.plot(ax=ax, color='blue', alpha=0.5, edgecolor='black', label=f"{burst}")
    ax.legend()
    ax.set_title(f"Burst : {burst} and ROI overlap")
    plt.show()
    

In [ ]:
if not burst_roi:
    results_folder = f'timeseries-results/{burst}'
else:
    results_folder = f'timeseries-results/{burst}_{roi_x}_{roi_y}'
os.makedirs(results_folder, exist_ok=True)

## Functions

In [ ]:
# Adapted from https://stackoverflow.com/questions/39785970/speckle-lee-filter-in-python
def lee_filter(img, size):
    """
    Applies the Lee filter to reduce speckle noise in an image.

    Parameters:
    img (ndarray): Input image to be filtered.
    size (int): Size of the uniform filter window.

    Returns:
    ndarray: The filtered image.
    """
    img_mean = uniform_filter(img, size)
    img_sqr_mean = uniform_filter(img**2, size)
    img_variance = img_sqr_mean - img_mean**2

    overall_variance = np.var(img)

    img_weights = img_variance / (img_variance + overall_variance)
    img_output = img_mean + img_weights * (img - img_mean)
    return img_output


# Define a function to apply the Lee filter to a DataArray
def apply_lee_filter(data_array, size=7):
    """
    Applies the Lee filter to the provided DataArray.

    Parameters:
    data_array (xarray.DataArray): The data array to be filtered.
    size (int): Size of the uniform filter window. Default is 7.

    Returns:
    xarray.DataArray: The filtered data array.
    """
    data_array_filled = data_array.fillna(0)
    filtered_data = xr.apply_ufunc(
        lee_filter,
        data_array_filled,
        kwargs={"size": size},
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask_gufunc_kwargs={"allow_rechunk": True},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[data_array.dtype],
    )
    filtered_data_masked = xr.where(np.isnan(data_array), np.nan, filtered_data)

    return filtered_data_masked


def slope_aspect(dem, dx=1.0, dy=1.0):
    """
    Compute slope and aspect from a 2D DEM array, handling nodata values.
    
    Parameters
    ----------
    dem : np.ndarray
        2D elevation array (NaNs represent nodata).
    dx : float
        Spatial resolution in x-direction.
    dy : float
        Spatial resolution in y-direction.
    
    Returns
    -------
    slope : np.ndarray
        Slope in degrees.
    aspect : np.ndarray
        Aspect in degrees (0 = North, clockwise).
    """
    # Mask invalid values
    dem = np.where(np.isfinite(dem), dem, np.nan)
    
    # Compute gradients
    dz_dy, dz_dx = np.gradient(dem, dy, dx)
    
    # Replace NaN gradients with 0 to avoid NaNs in slope/aspect
    dz_dx = np.nan_to_num(dz_dx)
    dz_dy = np.nan_to_num(dz_dy)
    
    # Slope in degrees
    slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)) * 180 / np.pi
    
    # Aspect in degrees
    aspect = np.arctan2(-dz_dx, -dz_dy) * 180 / np.pi
    aspect = np.where(np.isnan(dem), np.nan, (aspect + 360) % 360)
    
    return slope, aspect


def slope_aspect_timeseries(ds_dem, dx=1.0, dy=1.0):
    """
    Apply slope_aspect function to an xarray DataArray over time,
    preserving NaNs for nodata values.
    
    Parameters
    ----------
    ds_dem : xarray.DataArray
        DEM time series with dimensions ('time', 'y', 'x')
    dx, dy : float
        Spatial resolution
    
    Returns
    -------
    slopes : xarray.DataArray
        Slope for each timestep
    aspects : xarray.DataArray
        Aspect for each timestep
    """
    slopes, aspects = xr.apply_ufunc(
        slope_aspect,
        ds_dem,
        dx,
        dy,
        input_core_dims=[["y", "x"], [], []],
        output_core_dims=[["y", "x"], ["y", "x"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[ds_dem.dtype, ds_dem.dtype],
    )
    
    slopes = xr.DataArray(slopes, coords=ds_dem.coords, dims=ds_dem.dims, name="slope")
    aspects = xr.DataArray(aspects, coords=ds_dem.coords, dims=ds_dem.dims, name="aspect")
    
    return slopes, aspects

def keep_s3_prefix(file_list):
    return [f if f.startswith("s3://") else f"s3://{f}" for f in file_list]

# Extract timestamps from filenames (e.g., 20170405T050842)
def extract_time(fp):
    match = re.search(r"(\d{8}T\d{6})", fp)
    return pd.to_datetime(match.group(1)) if match else None

def load_xr_dataset(file_list):
    datasets = []
    for fp in file_list:
        da = rioxarray.open_rasterio(
            fp,
            masked=True,
            chunks=True,
        )
        da = da.squeeze().expand_dims(time=[extract_time(fp)])  # add time dim
        datasets.append(da)

        # Combine along the time dimension
    return xr.concat(datasets, dim="time")  # dims: ('time', 'y', 'x')

def preprocess_gamma0_xr_dataset(ds_gamma0, burst_shape):
    # Open each file lazily with rioxarray and assign a time coordinate
    ds_gamma0.name = "HH_gamma0"
    # trim the dataset withg burst shapefule
    burst_shape = burst_shape.to_crs(ds_gamma0.rio.crs)
    ds_gamma0 = ds_gamma0.rio.clip(burst_shape.geometry, crs=ds_gamma0.rio.crs, drop=True)
    ds_gamma0.odc.assign_crs(crs='EPSG:3031')
    # Apply Lee filter directly on the DataArray
    ds_gamma0["HH_gamma0_filtered"] = apply_lee_filter(ds_gamma0, size=5)
    # Convert to dB only for positive values
    ds_gamma0["HH_gamma0_db"] = xr.where(
        ds_gamma0.to_dataset(name='HH_gamma0')['HH_gamma0'] > 0,
        10 * np.log10(ds_gamma0.to_dataset(name='HH_gamma0')['HH_gamma0']),
        np.nan
    )
    ds_gamma0["HH_gamma0_filtered_db"] = 10 * np.log10(ds_gamma0.HH_gamma0_filtered)
    return ds_gamma0

def preprocess_dem_xr_dataset(ds_dem, burst_shape):
    ds_dem.name = "dem"
    burst_shape = burst_shape.to_crs(ds_dem.rio.crs)
    ds_dem = ds_dem.rio.clip(burst_shape.geometry, crs=ds_dem.rio.crs, drop=True)
    ds_dem.odc.assign_crs(crs='EPSG:3031')
    ds_dem = ds_dem.to_dataset(name="elevation")
    slopes, aspects = slope_aspect_timeseries(ds_dem['elevation'], dx=20, dy=20)
    ds_dem = ds_dem.assign({
        "slope": slopes,
        "aspect": aspects
    })
    return ds_dem


## Load in the xarrays

In [ ]:
# Create S3 filesystem (anonymous access)
fs = fsspec.filesystem("s3", anon=True)

# Base S3 folder (public)
static_base_path = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10/ga_s1_nrb_iw_hh_0/{burst}/"
timeseries_base_path = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10_TIMESERIES/ga_s1_nrb_iw_hh_0/{burst}/"

# Find all GeoTIFFs recursively
static_nrb_tif_files = fs.glob(f"{static_base_path}**/*HH-gamma0.tif")
static_dem_tif_files = fs.glob(f"{static_base_path}**/*digital-elevation-model.tif")
timeseries_nrb_tif_files = fs.glob(f"{timeseries_base_path}**/*HH-gamma0.tif")
timeseries_dem_tif_files = fs.glob(f"{timeseries_base_path}**/*digital-elevation-model.tif")

# Ensure each file keeps the s3:// prefix (sometimes fsspec strips it)
static_nrb_tif_files = keep_s3_prefix(static_nrb_tif_files)
static_dem_tif_files = keep_s3_prefix(static_dem_tif_files)
timeseries_nrb_tif_files = keep_s3_prefix(timeseries_nrb_tif_files)
timeseries_dem_tif_files = keep_s3_prefix(timeseries_dem_tif_files)

print(f"Found {len(static_nrb_tif_files)} static NRB TIFs:")
print(f"Found {len(static_dem_tif_files)} static DEM TIFs:")
print(f"Found {len(timeseries_nrb_tif_files)} timeseries NRB TIFs:")
print(f"Found {len(timeseries_dem_tif_files)} timeseries DEM TIFs:")

# get the shape for trimming the xarray
clip_shape = burst_roi_shape if burst_roi else burst_shape

# load and preprocess the gamma0 datasets
ds_static_nrb = load_xr_dataset(static_nrb_tif_files)
ds_static_nrb = preprocess_gamma0_xr_dataset(ds_static_nrb, clip_shape)
ds_timeseries_nrb = load_xr_dataset(timeseries_nrb_tif_files)
ds_timeseries_nrb = preprocess_gamma0_xr_dataset(ds_timeseries_nrb, clip_shape)

# load in the dem datasets
ds_static_dem = load_xr_dataset(static_dem_tif_files)
ds_static_dem = preprocess_dem_xr_dataset(ds_static_dem, clip_shape)
ds_timeseries_dem = load_xr_dataset(timeseries_dem_tif_files)
ds_timeseries_dem = preprocess_dem_xr_dataset(ds_timeseries_dem, clip_shape)

#ds_static_nrb.isel(time=1).HH_gamma0_filtered_db.plot.imshow(cmap="bone", vmin=-20, vmax=0, figsize=(12,5)) 

In [ ]:
make_animation = True
if make_animation:
    for i,ds in enumerate([ds_static_nrb,ds_timeseries_nrb,ds_static_dem,ds_timeseries_dem]):
        dem_type = ['REMA_10','REMA_10_TIMESERIES','REMA_10','REMA_10_TIMESERIES'][i]
        plot_var = ['HH_gamma0_filtered_db','HH_gamma0_filtered_db','elevation','elevation'][i]
        titles = ['Gamma0 (dB)', 'Gamma0 (dB)', 'Elevation (m)', 'Elevation (m)']
        if plot_var == 'HH_gamma0_filtered_db':
            imshow_kwargs={"cmap":"bone", "vmin":-20, "vmax":0}
            plot_ds = ds[plot_var].to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
            band = ds.name
        if plot_var == 'elevation':
            imshow_kwargs={}
            plot_ds = ds[plot_var].to_dataset(name='elevation').odc.assign_crs(crs='EPSG:3031')
            band = plot_var
        xr_animation(
            plot_ds,
            bands=band,
            output_path=f'{results_folder}/{burst}_{dem_type}_{band}.gif',
            width_pixels=1200,
            interval=300,
            show_date='%d %b %Y',
            show_text=f'{titles[i]} - burst: {burst}  dem_type: {dem_type}',
            show_colorbar=True,
            imshow_kwargs=imshow_kwargs,
            colorbar_kwargs={'colors': 'black'},
        )

## Animation of NRB differnces at each timestep

In [ ]:
# Compute difference at each timestep
nrb_diff_ts = ds_timeseries_nrb.HH_gamma0_filtered_db - ds_static_nrb.HH_gamma0_filtered_db
nrb_diff_ds = xr.Dataset({"HH_gamma0_filtered_db_diff": nrb_diff_ts})

In [ ]:
make_animation = True
#.to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
if make_animation:
    xr_animation(
        nrb_diff_ds.odc.assign_crs(crs='EPSG:3031'),
        bands="HH_gamma0_filtered_db_diff",
        output_path=f'{results_folder}/{burst}_gamma0_difference.gif',
        width_pixels=1200,
        interval=300,
        show_date='%d %b %Y',
        show_text=f'Gamma0 (dB) Difference - burst: {burst}',
        show_colorbar=True,
        imshow_kwargs={"cmap":"RdBu", "vmin":-2, "vmax":2},
        colorbar_kwargs={'colors': 'black'},
    )

## Animation of NRB differnces at each timestep

In [ ]:
# Compute difference at each timestep
dem_diff_ts = ds_timeseries_dem.elevation - ds_static_dem.elevation
dem_diff_ds = xr.Dataset({"elevation_m_diff": dem_diff_ts})

In [ ]:
make_animation = True
#.to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
if make_animation:
    xr_animation(
        dem_diff_ds.odc.assign_crs(crs='EPSG:3031'),
        bands="elevation_m_diff",
        output_path=f'{results_folder}/{burst}_dem_difference.gif',
        width_pixels=1200,
        interval=300,
        show_date='%d %b %Y',
        show_text=f'Elevation Difference (m) - burst: {burst}',
        show_colorbar=True,
        imshow_kwargs={"cmap":"RdBu"},
        colorbar_kwargs={'colors': 'black'},
    )

## Timeseries of radiometry and elevation

In [ ]:
static_nrb_mean_db = ds_static_nrb.HH_gamma0_db.mean(dim=["y", "x"], skipna=True)
timseries_nrb_mean_db = ds_timeseries_nrb.HH_gamma0_db.mean(dim=["y", "x"], skipna=True)
static_elevation_mean = ds_static_dem.elevation.mean(dim=["y", "x"])
timeseries_elevation_mean = ds_timeseries_dem.elevation.mean(dim=["y", "x"])
static_slope_mean = ds_static_dem.slope.mean(dim=["y", "x"])
timeseries_slope_mean = ds_timeseries_dem.slope.mean(dim=["y", "x"])
static_aspect_mean = ds_static_dem.aspect.mean(dim=["y", "x"])
timeseries_aspect_mean = ds_timeseries_dem.aspect.mean(dim=["y", "x"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# --- Left: NRB mean comparison ---
print(f'Plotting NRB')
static_nrb_mean_val = np.mean(static_nrb_mean_db.values)
ts_nrb_mean_val = np.mean(timseries_nrb_mean_db.values)
print(f'Static Mean (dB): {static_nrb_mean_val}')
print(f'Timeseries Mean (dB): {ts_nrb_mean_val}')
axes[0][0].plot(static_nrb_mean_db.time, static_nrb_mean_db, marker='o', label=f'Static NRB (u={static_nrb_mean_val:.4g})')
axes[0][0].plot(timseries_nrb_mean_db.time, timseries_nrb_mean_db, marker='x', label=f'Timeseries NRB (u={ts_nrb_mean_val:.4g})')
axes[0][0].set_title("NRB Mean Backscatter Over Time", color='red')
axes[0][0].set_xlabel("Time")
axes[0][0].set_ylabel("HH_gamma0 (dB)")
axes[0][0].legend()
axes[0][0].grid(True)

# --- Right: elevation mean comparison ---
print(f'PLotting elevation')
axes[0][1].plot(static_elevation_mean.time, static_elevation_mean, marker='o', label='Static DEM')
axes[0][1].plot(timeseries_elevation_mean.time, timeseries_elevation_mean, marker='x', label='Timeseries DEM')
axes[0][1].set_title("DEM Mean Elevation Over Time")
axes[0][1].set_xlabel("Time")
axes[0][1].set_ylabel("Elevation (m)")
axes[0][1].legend()
axes[0][1].grid(True)

# --- Right: slope mean comparison ---
print(f'Plotting slope')
axes[1][0].plot(static_slope_mean.time, static_slope_mean, marker='o', label='Static DEM')
axes[1][0].plot(timeseries_slope_mean.time, timeseries_slope_mean, marker='x', label='Timeseries DEM')
axes[1][0].set_title("DEM Mean Slope Over Time")
axes[1][0].set_xlabel("Time")
axes[1][0].set_ylabel("Slope (degrees)")
axes[1][0].legend()
axes[1][0].grid(True)

# --- Right: aspect mean comparison ---
print(f'Plotting aspect')
axes[1][1].plot(static_aspect_mean.time, static_aspect_mean, marker='o', label='Static DEM')
axes[1][1].plot(timeseries_aspect_mean.time, timeseries_aspect_mean, marker='x', label='Timeseries DEM')
axes[1][1].set_title("DEM Mean Aspect Over Time")
axes[1][1].set_xlabel("Time")
axes[1][1].set_ylabel("Aspect (degrees)")
axes[1][1].legend()
axes[1][1].grid(True)

plt.suptitle(f'Burst ID : {burst}')
plt.tight_layout()
plt.savefig(f'{results_folder}/{burst}_linear_plots.png')
plt.show()

## Plot timeseries max difference maps

In [ ]:
# Compute difference: second to last timestep minus first timestep
for param in ['slope','aspect','elevation']:
    ey = ds_timeseries_dem[param].time.dt.year.isel(time=-2).item()
    sy = ds_timeseries_dem[param].time.dt.year.isel(time=1).item()
    diff = ds_timeseries_dem[param].isel(time=-2) - ds_timeseries_dem[param].isel(time=1)
    plt.figure(figsize=(12, 5))
    diff.plot.imshow(cmap="RdBu")
    plt.title(f"Burst {burst} {param.upper()} Difference: 2023 minus 2015")
    plt.savefig(f'{results_folder}/{burst}_{dem_type}_{band}_{param.upper()}_diff_{ey}_minus_{sy}.png')
    plt.show()

## Optical Flow

In [ ]:
def run_optical_flow(ds, downscale_factor):
    ds_course = ds.coarsen(x=downscale_factor, y=downscale_factor, boundary="trim").mean()
    ds_flow = xr_optical_flow(ds_course.HH_gamma0_filtered_db.fillna(0), method="tvl1", baseline="dynamic")
    valid_mask = ds.notnull().all(dim="time")
    ds_flow = ds_flow.where(valid_mask)
    # Compute timestep means
    mean_mag = ds_flow.magnitude.mean(dim=("x", "y"))*20*downscale_factor
    mean_u   = ds_flow.u.mean(dim=("x", "y"))*20*downscale_factor
    mean_v   = ds_flow.v.mean(dim=("x", "y"))*20*downscale_factor
    # Compute timestep means
    med_mag = ds_flow.magnitude.median(dim=("x", "y"))*20*downscale_factor
    med_u   = ds_flow.u.median(dim=("x", "y"))*20*downscale_factor
    med_v   = ds_flow.v.median(dim=("x", "y"))*20*downscale_factor
    # Build a clean table
    df = xr.Dataset({
        "time": ds_flow.time,
        "mean magnitude": mean_mag,
        "mean u": mean_u,
        "mean v": mean_v,
        "median magnitude": med_mag,
        "median u": med_u,
        "median v": med_v
    }).to_dataframe().reset_index().drop(columns=["band", "spatial_ref"])
    return df, ds_flow

def run_optical_flow_between_timeseries(ds1, ds2, var="HH_gamma0_filtered_db", downscale_factor=3):
    """
    Run optical flow between matching timesteps of ds1 and ds2.
    
    Parameters
    ----------
    ds1, ds2 : xarray.Dataset
        Input datasets with the same spatial dimensions and times.
    var : str
        Variable name to use for optical flow.
    downscale_factor : int
        Factor for coarsening the data.
    
    Returns
    -------
    ds_flow_all : xarray.Dataset
        Optical flow results concatenated along 'time'.
    """

    times = ds1.time.values
    flow_list = []

    for i,t in enumerate(times):
        # Extract single timestep from each dataset
        print(f"time {i+1} of {len(times)} : {t}")
        da1 = ds1[var].sel(time=t)
        da2 = ds2[var].sel(time=t)
        # Coarsen both arrays
        da1_c = da1.coarsen(x=downscale_factor, y=downscale_factor, boundary="trim").mean()
        da2_c = da2.coarsen(x=downscale_factor, y=downscale_factor, boundary="trim").mean()
        # Combine into a 2-step DataArray along a new 'time' dimension
        da_pair = xr.concat([da1_c, da2_c], dim="time")
        # Compute optical flow between first and second step
        ds_flow = xr_optical_flow(da_pair, baseline="first", method="tvl1")
        # Assign original timestep as coordinate
        ds_flow = ds_flow.assign_coords(time=[t])
        valid_mask = da_pair.notnull().all(dim="time")
        ds_flow = ds_flow.where(valid_mask)
        flow_list.append(ds_flow)

    # Combine all timesteps into one Dataset
    ds_flow_all = xr.concat(flow_list, dim="time")
     # Compute timestep means
    mean_mag = ds_flow_all.magnitude.mean(dim=("x", "y"))*20*downscale_factor
    mean_u   = ds_flow_all.u.mean(dim=("x", "y"))*20*downscale_factor
    mean_v   = ds_flow_all.v.mean(dim=("x", "y"))*20*downscale_factor
    # Compute timestep means
    med_mag = ds_flow_all.magnitude.median(dim=("x", "y"))*20*downscale_factor
    med_u   = ds_flow_all.u.median(dim=("x", "y"))*20*downscale_factor
    med_v   = ds_flow_all.v.median(dim=("x", "y"))*20*downscale_factor
    # Build a clean table
    df = xr.Dataset({
        "time": ds_flow_all.time,
        "mean magnitude": mean_mag,
        "mean u": mean_u,
        "mean v": mean_v,
        "median magnitude": med_mag,
        "median u": med_u,
        "median v": med_v
    }).to_dataframe().reset_index().drop(columns=["band", "spatial_ref"])
    return df, ds_flow_all

def plot_optical_flow(
        ds_flow_to_plot, 
        base_plot_arr, 
        savepath, 
        year_label="",
        base_plot_label="",
        base_plot_cmap="RdBu",
        vrange=[]
        ):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    # Sub-sample array for clearer quiver plot
    ds_flow_coarse = ds_flow_to_plot.coarsen({"x": 35, "y": 35}, boundary="trim").median()
    # Flip vertical axis to avoid xarray issue where negative Y
    # coordinates cause vectors to be inverted
    ds_flow_coarse["v"] = -ds_flow_coarse.v
    # Plot background rate of vertical change image
    base_plot_arr.plot(
        ax=ax,
        cmap=base_plot_cmap,
        cbar_kwargs={"label": base_plot_label},
        vmin=None if not vrange else vrange[0],
        vmax=None if not vrange else vrange[1],
    )
    # Add a basemap for context
    ctx.add_basemap(
        ax,
        source=ctx.providers.Esri.WorldImagery,
        crs="EPSG:3031",
        attribution="Esri WorldImagery",
        attribution_size=1,
        alpha=0.7,
    )
    # Add quiver plot directly from xarray
    quiver = ds_flow_coarse.plot.quiver(
        x="x",
        y="y",
        u="u",
        v="v",
        ax=ax,
        color="black",
        pivot="mid",
        add_guide=False,
        width=0.0015
    )
    # Add key and plot title
    ax.quiverkey(quiver, 0.85, 0.85, 1, f" {20*downscale_factor} m / year")
    # add test to plot for year
    ax.text(
        0.85, 0.95,                 # just below the quiver key
        year_label,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=14
    )

    ax.set_title(f"Burst ID : {burst}, dem_type : {dem_type} Surface flow (optical flow vectors)")
    ax.set_axis_off()
    fig.tight_layout();
    plt.savefig(savepath);

def plot_optical_flow_for_year(ds, base_plot_arr, year, savepath, base_plot_label, base_plot_cmap="RdBu",vrange=[]):
    ds_year = ds.sel(time=str(year))
    ds_year_median = ds_year.median(dim="time")
    plot_optical_flow(ds_year_median, base_plot_arr, savepath, year_label=f"{year}",base_plot_label=base_plot_label,base_plot_cmap=base_plot_cmap, vrange=vrange)


In [ ]:
downscale_factor = 3 # factor to downsample the image by i.e. 20m -> 60m (x3)
print(f'Calculating static optical flow')
df_static_flow_summary, ds_static_nrb_flow = run_optical_flow(ds_static_nrb, downscale_factor)
print(f'Calculating timeseries optical flow')
df_timeseries_flow_summary, ds_timeseries_nrb_flow = run_optical_flow(ds_timeseries_nrb, downscale_factor)

In [ ]:
print(f'Optical flow between timesteps')
df_compare_flow_summary, ds_compare_nrb_flow = run_optical_flow_between_timeseries(ds_static_nrb, ds_timeseries_nrb, var="HH_gamma0_filtered_db", downscale_factor=3)

In [ ]:
df_static_flow_summary

In [ ]:
print('REMA_10 Static DEM')
print(df_static_flow_summary)
print('\nREMA_10 Timeseries DEM')
print(df_timeseries_flow_summary)
print('\nVelocity overestimation (m/year)')
cols = ["mean magnitude", "mean u", "mean v","median magnitude", "median u", "median v"]
overestimate = ((df_static_flow_summary[cols].values - df_timeseries_flow_summary[cols].values))
overestimate = pd.DataFrame(overestimate, index=df_timeseries_flow_summary.index, columns=cols)
print(overestimate)
avg_overestimate = ((df_static_flow_summary.mean()[cols] - df_timeseries_flow_summary.mean()[cols]))
avg_static = df_static_flow_summary.mean()[cols]
avg_timeseries = df_timeseries_flow_summary.mean()[cols]
df_avg = xr.Dataset({
    "REMA_10_TIMESERIES": avg_timeseries,
    "REMA_10": avg_static,
    "Overestimation": avg_overestimate,
})
print('\nAverage velocity (m/year)')
print(df_avg)
print(f'\nDifference between matching time pairs : REMA_10 - REMA_10_TIMESERIES')
df_compare_flow_summary


## Median Plots

In [ ]:
print(f'making plots')
savepath = f'{results_folder}/{burst}_REMA_10_median_optical_flow.png'
base_label = f"{sy} to {ey} timeseries elevation change (m)"
dem_type = 'REMA_10'
plot_optical_flow(
    ds_static_nrb_flow.median(dim="time"), 
    diff,
    savepath, 
    year_label="Timeseries median",
    base_plot_label=base_label
    )
df_static_flow_summary.to_csv(savepath.replace('png','csv'))
savepath = f'{results_folder}/{burst}_REMA_10_TIMESERIES_median_optical_flow.png'
dem_type = 'REMA_10_TIMESERIES'
plot_optical_flow(
    ds_timeseries_nrb_flow.median(dim="time"), 
    diff,
    savepath, 
    year_label="Timeseries median",
    base_plot_label=base_label
    )
df_timeseries_flow_summary.to_csv(savepath.replace('png','csv'))

In [ ]:
savepath = f'{results_folder}/{burst}_median_nrb_diff_optical_flow.png'
dem_type = ''
plot_optical_flow(
    ds_compare_nrb_flow.median(dim="time"), 
    dem_diff_ds.elevation_m_diff.median(dim="time"),
    savepath, 
    year_label="Timeseries median",
    base_plot_label="Elevation Difference"
    )
df_compare_flow_summary.to_csv(savepath.replace('png','csv'))

## Yearly plots into gifs

In [ ]:
for dem_type, ds_nrb_flow, ds_nrb in [('REMA_10',ds_static_nrb_flow,ds_static_nrb),('REMA_10_TIMESERIES',ds_timeseries_nrb_flow, ds_timeseries_nrb)]:
    years = sorted(set(pd.to_datetime(ds_nrb_flow.time.values).year))
    year_pngs = []
    os.makedirs(f"{results_folder}/optical_flow", exist_ok=True)
    print(f"Generating per-year plots for {dem_type}...")
    for i,yr in enumerate(years):
        print(yr)
        savepath = f"{results_folder}/optical_flow/{burst}_{dem_type}_{yr}_optical_flow.png"
        plot_optical_flow_for_year(
            ds_nrb_flow, 
            ds_nrb.isel(time=i).HH_gamma0_filtered_db, 
            yr, 
            savepath, 
            base_plot_label = f"{sy} to {ey} timeseries elevation change (m)",
            base_plot_cmap="bone",
            vrange=[-20,0]
        )
        year_pngs.append(savepath)
        # if i == 2:
        # break

    gif_path = f"{results_folder}/{burst}_{dem_type}_optical_flow_timeseries.gif"
    # Open first frame
    frames = [Image.open(p) for p in year_pngs]
    # Save as GIF
    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=300,     # ms per frame
        loop=0           # infinite loop
    )
    print(f"GIF saved to: {gif_path}")

In [ ]:
years = sorted(set(pd.to_datetime(ds_compare_nrb_flow.time.values).year))
year_pngs = []
os.makedirs(f"{results_folder}/optical_flow", exist_ok=True)
print(f"Generating per-year plots for the offset between nrb gifs...")
dem_type = ''
for i,yr in enumerate(years):
    print(yr)
    savepath = f"{results_folder}/optical_flow/{burst}_{yr}_nrb_diff_optical_flow.png"
    plot_optical_flow_for_year(
        ds_compare_nrb_flow, 
        dem_diff_ds.isel(time=i).elevation_m_diff,
        yr, 
        savepath, 
        base_plot_label = f"Elevation Difference")
    year_pngs.append(savepath)

gif_path = f"{results_folder}/{burst}_nrb_diff_optical_flow_timeseries.gif"
# Open first frame
frames = [Image.open(p) for p in year_pngs]
# Save as GIF
frames[0].save(
    gif_path,
    save_all=True,
    append_images=frames[1:],
    duration=300,     # ms per frame
    loop=0           # infinite loop
)
print(f"GIF saved to: {gif_path}")

# Combine GIFS

In [ ]:
def uniform_frames(gif_path, frame_interval=100):
    """Convert GIF to a list of frames at uniform intervals (ms). As there are
    'empty' repeated frames in the winter months of the .gif, we need to ensure
    the frames and intervals are consistent"""
    gif = Image.open(gif_path)
    frames_list = [frame.convert("RGBA") for frame in ImageSequence.Iterator(gif)]
    
    # Original frame durations
    durations = [frame.info.get('duration', gif.info['duration']) for frame in ImageSequence.Iterator(gif)]
    cum_times = np.cumsum(durations)
    total_time = cum_times[-1]
    
    # Number of uniform frames
    times = np.arange(0, total_time, frame_interval)
    
    uniform_frames = []
    f_idx = 0
    for t in times:
        # Advance to correct frame for this time
        while f_idx < len(cum_times) - 1 and t >= cum_times[f_idx]:
            f_idx += 1
        uniform_frames.append(frames_list[f_idx])
    
    return uniform_frames

def combine_gif_frames(frames_list, orientation="horizontal"):
    """
    Combine multiple sets of frames into a single animated GIF.

    Parameters
    ----------
    frames_list : list of lists of PIL.Image
        Each element is a list of frames to combine (must all have same length).
    orientation : str, optional
        "horizontal" for side-by-side, "vertical" for stacked. Default is "horizontal".

    Returns
    -------
    PIL.Image
        The first frame of the combined GIF (use .save with save_all to write full GIF).
    list of PIL.Image
        All combined frames.
    """
    num_frames = len(frames_list[0])
    combined_frames = []

    for i in range(num_frames):
        # get current frame from each set
        current_frames = [frames[i] for frames in frames_list]

        if orientation == "horizontal":
            total_width = sum(f.width for f in current_frames)
            max_height = max(f.height for f in current_frames)
            new_frame = Image.new("RGBA", (total_width, max_height))
            x_offset = 0
            for f in current_frames:
                new_frame.paste(f, (x_offset, 0))
                x_offset += f.width

        elif orientation == "vertical":
            max_width = max(f.width for f in current_frames)
            total_height = sum(f.height for f in current_frames)
            new_frame = Image.new("RGBA", (max_width, total_height))
            y_offset = 0
            for f in current_frames:
                new_frame.paste(f, (0, y_offset))
                y_offset += f.height
        else:
            raise ValueError("orientation must be 'horizontal' or 'vertical'")

        combined_frames.append(new_frame)

    return combined_frames

combine = True

if combine:

    # Load uniform frames for NRB
    frames1 = uniform_frames(f"{results_folder}/{burst}_REMA_10_HH_gamma0.gif", frame_interval=80)
    frames2 = uniform_frames(f"{results_folder}/{burst}_REMA_10_TIMESERIES_HH_gamma0.gif", frame_interval=80)
    frames3 = uniform_frames(f"{results_folder}/{burst}_gamma0_difference.gif", frame_interval=80)
    combined_frames = combine_gif_frames([frames1, frames2, frames3], orientation='vertical')
    # Save combined GIF
    combined_frames[0].save(
        f"{results_folder}/{burst}_gamma0_comparison.gif",
        save_all=True,
        append_images=combined_frames[1:],
        duration=80,  # uniform frame duration
        loop=0
    )

    # Load uniform frames for DEM
    frames1 = uniform_frames(f"{results_folder}/{burst}_REMA_10_elevation.gif", frame_interval=80)
    frames2 = uniform_frames(f"{results_folder}/{burst}_REMA_10_TIMESERIES_elevation.gif", frame_interval=80)
    frames3 = uniform_frames(f"{results_folder}/{burst}_dem_difference.gif", frame_interval=80)
    combined_frames = combine_gif_frames([frames1, frames2, frames3], orientation='vertical')
    # Save combined GIF
    combined_frames[0].save(
        f"{results_folder}/{burst}_dem_comparison.gif",
        save_all=True,
        append_images=combined_frames[1:],
        duration=80,  # uniform frame duration
        loop=0
    )
